In [ ]:
# NOTE: These parsers were built for Google Ads, TikTok Ads and TikTok Organic APIs.
# They are no longer in use in the current pipeline but kept for reference.

def parse_google_ads(raw_data):
    """
    Parses raw Google Ads API response into a list of tuples with
    campaign, ad group and performance metrics per day.
    """
    rows = []
    for g in raw_data:
        metrics = g.get("metrics", {})
        campaign = g.get("campaign", {})
        ad_group = g.get("adGroup", {})
        segments = g.get("segments", {})
        rows.append((
            str(segments.get("date", "")),
            "google",
            str(campaign.get("name", "")),
            str(ad_group.get("name", "")),
            str(g.get("adGroupAd", {}).get("ad", {}).get("name", "")),
            int(metrics.get("impressions", 0)),
            0,
            int(metrics.get("clicks", 0)),
            int(metrics.get("costMicros", 0)) / 1000000,
            int(metrics.get("clicks", 0)),
            0,
            0,
            int(float(metrics.get("conversions", 0)))
        ))
    return rows


def parse_tiktok_ads(raw_data):
    """
    Parses raw TikTok Ads API response into a list of tuples with
    ad-level dimensions and performance metrics per day.
    """
    rows = []
    for t in raw_data:
        metrics = t.get("metrics", {})
        dimensions = t.get("dimensions", {})
        rows.append((
            str(dimensions.get("stat_time_day", "")),
            "tiktok",
            str(dimensions.get("campaign_name", "")),
            str(dimensions.get("adgroup_name", "")),
            str(dimensions.get("ad_name", "")),
            int(metrics.get("impressions", 0)),
            int(metrics.get("reach", 0)),
            int(metrics.get("clicks", 0)),
            float(metrics.get("spend", 0)),
            int(metrics.get("clicks", 0)),
            0,
            0,
            int(metrics.get("conversions", 0))
        ))
    return rows


def parse_tiktok_organic(raw_data):
    """
    Parses raw TikTok Organic API response into a list of tuples with
    video metadata and engagement metrics.
    """
    rows = []
    for t in raw_data:
        rows.append((
            "tiktok",
            str(t.get("id", "")),
            str(t.get("title", "")),
            str(t.get("create_time", "")),
            str(t.get("share_url", "")),
            int(t.get("share_count", 0)),
            int(t.get("like_count", 0)),
            int(t.get("comment_count", 0)),
            int(t.get("reach_count", 0)),
            int(t.get("view_count", 0))
        ))
    return rows

In [ ]:
def parse_tiendanube_orders(raw_data):
    """
    Parses raw Tiendanube Orders API response into a list of tuples.
    Expands each order by product line. If an order has no products, 
    inserts a single row with order-level totals.
    """
    rows = []
    for o in raw_data:
        order_id = str(o.get("id", ""))
        email = str(o.get("customer", {}).get("email", ""))
        created_at = str(o.get("created_at", ""))
        currency = str(o.get("currency", ""))
        financial_status = str(o.get("payment_status", ""))
        fulfillment_status = str(o.get("shipping_status", "") or "unfulfilled")
        total_discount = float(o.get("discount", 0))
        
        products = o.get("products", [])
        
        if len(products) > 0:
            for item in products:
                rows.append((
                    "tiendanube",
                    order_id,
                    str(item.get("product_id", "")),
                    str(item.get("name", "")),
                    email,
                    created_at,
                    float(item.get("price", 0)),
                    float(item.get("price", 0)) * int(item.get("quantity", 1)),
                    total_discount,
                    currency,
                    int(item.get("quantity", 1)),
                    financial_status,
                    fulfillment_status,
                    len(products)
                ))
        else:
            rows.append((
                "tiendanube",
                order_id,
                "",
                "",
                email,
                created_at,
                float(o.get("total", 0)),
                float(o.get("total", 0)),
                total_discount,
                currency,
                0,
                financial_status,
                fulfillment_status,
                0
            ))
    return rows


def parse_tiendanube_products(raw_data):
    """
    Parses raw Tiendanube Products API response into a list of tuples.
    Extracts price and stock from the first variant. Handles multilingual product names.
    """
    rows = []
    for p in raw_data:
        price = 0.0
        inv = 0
        variants = p.get("variants", [])
        if len(variants) > 0:
            price = float(variants[0].get("price", 0))
            inv = int(variants[0].get("stock", 0))
        name = p.get("name", {})
        title = name.get("es", "") if isinstance(name, dict) else str(name)
        rows.append((
            "tiendanube",
            str(p.get("id", "")),
            title,
            str(p.get("product_type", "")),
            str(p.get("brand", "")),
            price,
            "active" if p.get("published", False) else "draft",
            inv
        ))
    return rows


def parse_tiendanube_customers(raw_data):
    """
    Parses raw Tiendanube Customers API response into a list of tuples
    with customer identity, location and purchase summary.
    """
    rows = []
    for c in raw_data:
        addr = c.get("default_address", {}) or {}
        rows.append((
            "tiendanube",
            str(c.get("id", "")),
            str(c.get("email", "")),
            str(c.get("name", "")),
            str(c.get("last_name", "")),
            str(addr.get("city", "")),
            str(addr.get("country", "")),
            int(c.get("total_orders", 0)),
            float(c.get("total_spent", 0))
        ))
    return rows

In [ ]:
# Maps each platform to its raw data path in the lakehouse and its corresponding parser function.

PARSERS_ADS = {
    "meta": ("meta/ad_insights", parse_meta_ads),
    "google_ads": ("google/ad_insights", parse_google_ads),
    "tiktok": ("tiktok/ad_insights", parse_tiktok_ads)
}

PARSERS_ORGANIC = {
    "meta": ("meta/organic_posts_enriched", parse_meta_organic),
    "tiktok": ("tiktok/organic_posts", parse_tiktok_organic)
}

PARSERS_ECOMMERCE = {
    "shopify": {
        "orders": ("shopify/orders", parse_shopify_orders),
        "products": ("shopify/products", parse_shopify_products),
        "customers": ("shopify/customers", parse_shopify_customers)
    },
    "tiendanube": {
        "orders": ("tiendanube/orders", parse_tiendanube_orders),
        "products": ("tiendanube/products", parse_tiendanube_products),
        "customers": ("tiendanube/customers", parse_tiendanube_customers)
    }
}